<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/fluidflow/neqsim_fenicsx_fem_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeqSim + FEniCSx: local FEM from process-model boundary conditions

This advanced example connects **NeqSim Java master → fluid properties / hydrate equilibrium → 1D pipeline screen → FEniCSx axisymmetric wall/insulation FEM → cooldown / hydrate margin → thermo-elastic stress**. The case is a wet-gas subsea line with a local degraded-insulation patch. It is a teaching and screening model, not a piping-code assessment.

In [ ]:
import hashlib, importlib.metadata, os, shutil, subprocess, sys
from pathlib import Path
NEQSIM_SOURCE_REF='master'
os.environ['NEQSIM_JVM_AUTOSTART']='0'
def rq(cmd,cwd=None):
    return subprocess.run(cmd,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,check=True).stdout
rq([sys.executable,'-m','pip','install','-q','neqsim','scipy','pyvista'])
try:
    import dolfinx
except ImportError:
    p=Path('/tmp/fenicsx.sh'); rq(['wget','-q','https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh','-O',str(p)]); rq(['bash',str(p)])
src=Path('/content/neqsim-java')
if src.exists(): shutil.rmtree(src)
rq(['git','clone','--depth','1','--branch',NEQSIM_SOURCE_REF,'https://github.com/equinor/neqsim.git',str(src)])
commit=rq(['git','-C',str(src),'rev-parse','HEAD']).strip()
rq(['./mvnw','-q','-DskipTests','-P','shade','package'],cwd=src)
jar=sorted((src/'target').glob('neqsim-*-shaded.jar'))[-1]
jar_sha=hashlib.sha256(jar.read_bytes()).hexdigest()
import jpype
jpype.addClassPath(str(jar))
if not jpype.isJVMStarted(): jpype.startJVM('-Xrs',convertStrings=False,interrupt=False)
SystemSrkCPAstatoil=jpype.JClass('neqsim.thermo.system.SystemSrkCPAstatoil')
ThermodynamicOperations=jpype.JClass('neqsim.thermodynamicoperations.ThermodynamicOperations')
SurfCooldownAnalyzer=jpype.JClass('neqsim.pvtsimulation.flowassurance.SurfCooldownAnalyzer')
loc=str(SurfCooldownAnalyzer.class_.getProtectionDomain().getCodeSource().getLocation())
assert jar.name in loc
print('NeqSim master commit:',commit); print('JAR SHA-256:',jar_sha); print('Main-only class source:',loc); print('DOLFINx:',importlib.metadata.version('fenics-dolfinx'))

## 1. NeqSim fluid → heat-transfer boundary

NeqSim supplies phase equilibrium, density, viscosity, thermal conductivity and mass-specific heat capacity. A Gnielinski correlation converts these to an internal HTC; a 1D energy balance establishes the local bulk boundary for FEM.

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc
import numpy as np, pandas as pd, matplotlib.pyplot as plt, ufl, pyvista as pv
from dolfinx import fem, mesh, plot
from dolfinx.fem.petsc import LinearProblem
pv.OFF_SCREEN=True
Tin,Pin,Pout,Tsea=60.,80.,70.,4.; Di,ts,ti=0.254,0.0127,0.050
ri,rs,ro=Di/2,Di/2+ts,Di/2+ts+ti; Lpipe,v=20000.,5.
ks,kins,kbad=50.,0.17,0.70; ho=300.; localL=4.; z0,z1=1.5,2.5
rhos,cps,rhoi,cpi=7850.,500.,600.,1700.; depth,rhosea,g=300.,1025.,9.80665
dry={'nitrogen':.01,'CO2':.02,'methane':.85,'ethane':.07,'propane':.03,'i-butane':.006,'n-butane':.008,'i-pentane':.002,'n-pentane':.002,'n-hexane':.002}; water=2e-4
z={k:x*(1-water) for k,x in dry.items()}; z['water']=water
def fluid(T,P,hydrate=False):
    f=SystemSrkCPAstatoil(T+273.15,P)
    for n,x in z.items(): f.addComponent(n,float(x))
    f.setMixingRule(10); f.setMultiPhaseCheck(True); f.setHydrateCheck(bool(hydrate))
    ThermodynamicOperations(f).TPflash(); f.initPhysicalProperties(); return f
def props(T,P):
    ph=fluid(T,P).getPhase('gas')
    return {'rho':float(ph.getDensity('kg/m3')),'mu':float(ph.getViscosity('kg/msec')),'k':float(ph.getThermalConductivity('W/mK')),'cp':float(ph.getCp('J/kgK'))}
p=props(Tin,Pin); Re=p['rho']*v*Di/p['mu']; Pr=p['cp']*p['mu']/p['k']; fd=(.79*np.log(Re)-1.64)**-2; Nu=(fd/8)*(Re-1000)*Pr/(1+12.7*np.sqrt(fd/8)*(Pr**(2/3)-1)); hi=Nu*p['k']/Di
def Rprime(k=kins): return 1/(hi*2*np.pi*ri)+np.log(rs/ri)/(2*np.pi*ks)+np.log(ro/rs)/(2*np.pi*k)+1/(ho*2*np.pi*ro)
x=np.linspace(0,Lpipe,121); P=np.linspace(Pin,Pout,len(x)); T=np.empty_like(x); T[0]=Tin; A=np.pi*ri**2
for j in range(len(x)-1):
    q=props(T[j],P[j]); mdot=q['rho']*v*A; T[j+1]=T[j]-(1/Rprime())*(T[j]-Tsea)/(mdot*q['cp'])*(x[j+1]-x[j])
Tb=float(np.interp(Lpipe/2,x,T)); Pb=float(np.interp(Lpipe/2,x,P))
print(pd.DataFrame({'rho':[p['rho']],'mu':[p['mu']],'k':[p['k']],'Cp':[p['cp']],'Re':[Re],'Pr':[Pr],'hi':[hi]})); print('10 km boundary:',Tb,'degC',Pb,'bara')
plt.plot(x/1000,T); plt.xlabel('Distance [km]'); plt.ylabel('Temperature [degC]'); plt.grid(); plt.show()

## 2. FEniCSx local thermal field

The local $z-r$ model is axisymmetric, so every weak-form integral is weighted by $2\pi r$. The inner wall convects to the NeqSim bulk gas; the outer insulation convects to seawater. A 1 m interval uses degraded insulation conductivity.

In [ ]:
def tags_rect(m,rmin,rmax):
    fd=m.topology.dim-1; a=mesh.locate_entities_boundary(m,fd,lambda x:np.isclose(x[1],rmin)); b=mesh.locate_entities_boundary(m,fd,lambda x:np.isclose(x[1],rmax)); e=np.hstack([a,b]).astype(np.int32); val=np.hstack([np.ones(len(a),np.int32),2*np.ones(len(b),np.int32)]); o=np.argsort(e); return mesh.meshtags(m,fd,e[o],val[o])
def thermal(nx=80,nr=18,defect=True):
    m=mesh.create_rectangle(MPI.COMM_WORLD,np.array([[0.,ri],[localL,ro]]),[nx,nr],cell_type=mesh.CellType.triangle); tag=tags_rect(m,ri,ro); ds=ufl.Measure('ds',domain=m,subdomain_data=tag); dx=ufl.Measure('dx',domain=m); V=fem.functionspace(m,('Lagrange',1)); u=ufl.TrialFunction(V); w=ufl.TestFunction(V); X=ufl.SpatialCoordinate(m); r=X[1]
    kb=ufl.conditional(ufl.le(r,rs),ks,kins); bad=ufl.And(ufl.And(ufl.ge(X[0],z0),ufl.le(X[0],z1)),ufl.gt(r,rs)); k=ufl.conditional(bad,kbad,kb) if defect else kb
    a=k*ufl.inner(ufl.grad(u),ufl.grad(w))*2*np.pi*r*dx+hi*u*w*2*np.pi*r*ds(1)+ho*u*w*2*np.pi*r*ds(2); L=hi*(Tb+273.15)*w*2*np.pi*r*ds(1)+ho*(Tsea+273.15)*w*2*np.pi*r*ds(2)
    uh=LinearProblem(a,L,petsc_options_prefix='heat_',petsc_options={'ksp_type':'preonly','pc_type':'lu'}).solve(); return m,tag,uh
m,tag,Th=thermal(); c=Th.function_space.tabulate_dof_coordinates(); Tc=Th.x.array-273.15; mask=np.isclose(c[:,1],ri); zz=c[mask,0]; ww=Tc[mask]; o=np.argsort(zz)
plt.plot(zz[o],ww[o]); plt.axvspan(z0,z1,alpha=.2); plt.xlabel('Local z [m]'); plt.ylabel('Inner wall T [degC]'); plt.grid(); plt.show()
# independent verification
mu,tu,U=thermal(defect=False); cu=U.function_space.tabulate_dof_coordinates(); Tui=(U.x.array-273.15)[np.isclose(cu[:,1],ri)].mean(); q=(Tb-Tsea)/Rprime(); Ta=Tb-q/(hi*2*np.pi*ri); mf,tf,F=thermal(140,28,True); cf=F.function_space.tabulate_dof_coordinates(); fine=(F.x.array-273.15)[np.isclose(cf[:,1],ri)].min(); assert abs(Tui-Ta)<0.2 and abs(fine-ww.min())<0.35
cells,types,xyz=plot.vtk_mesh(Th.function_space); grid=pv.UnstructuredGrid(cells,types,xyz); grid.point_data['T [degC]']=Tc; pl=pv.Plotter(off_screen=True); pl.add_mesh(grid,scalars='T [degC]'); pl.view_xy(); pl.screenshot('/tmp/femT.png'); from IPython.display import Image,display; display(Image('/tmp/femT.png'))

## 3. Hydrate no-touch time: global NeqSim screen versus local FEM cold spot

NeqSim calculates hydrate equilibrium at the local pressure and its `SurfCooldownAnalyzer` gives the uniform-pipe lumped cooldown. The local FEM resolves the colder wall around damaged insulation.

In [ ]:
h=fluid(Tb,Pb,True); ThermodynamicOperations(h).hydrateFormationTemperature(); Thyd=float(h.getTemperature('C')); target=Thyd+3.
s=SurfCooldownAnalyzer(fluid(Tb,Pb)); s.setInternalDiameter(Di); s.setWallThickness(ts); s.setInsulationThickness(ti); s.setInsulationConductivity(kins); s.setExternalHTC(ho); s.setSeabedTemperature(Tsea); s.setOperatingTemperature(Tb); s.setHydrateMargin(3.); s.setTotalTimeHours(48.); s.calculate(); print('Hydrate equilibrium:',Thyd,'degC; lumped no-touch:',float(s.getNoTouchTimeHours()),'h')
# Local transient solid FEM + lumped fluid inventory, implicit Euler.
def cooldown(dt=1800.,hours=48.):
    m=mesh.create_rectangle(MPI.COMM_WORLD,np.array([[0.,ri],[localL,ro]]),[50,14],cell_type=mesh.CellType.triangle); tag=tags_rect(m,ri,ro); ds=ufl.Measure('ds',domain=m,subdomain_data=tag); dx=ufl.Measure('dx',domain=m); V=fem.functionspace(m,('Lagrange',1)); X=ufl.SpatialCoordinate(m); r=X[1]; kb=ufl.conditional(ufl.le(r,rs),ks,kins); bad=ufl.And(ufl.And(ufl.ge(X[0],z0),ufl.le(X[0],z1)),ufl.gt(r,rs)); k=ufl.conditional(bad,kbad,kb); C=ufl.conditional(ufl.le(r,rs),rhos*cps,rhoi*cpi); un=fem.Function(V); un.x.array[:]=Tb+273.15; u=ufl.TrialFunction(V); w=ufl.TestFunction(V); B=fem.Constant(m,PETSc.ScalarType(Tb+273.15)); S=fem.Constant(m,PETSc.ScalarType(Tsea+273.15)); a=C*u*w*2*np.pi*r*dx+dt*k*ufl.inner(ufl.grad(u),ufl.grad(w))*2*np.pi*r*dx+dt*hi*u*w*2*np.pi*r*ds(1)+dt*ho*u*w*2*np.pi*r*ds(2); gp=props(Tb,Pb); fluidC=gp['rho']*np.pi*ri**2*localL*gp['cp']; t=[0.]; bulk=[Tb]; wall=[Tb]
    for n in range(int(hours*3600/dt)):
        B.value=PETSc.ScalarType(bulk[-1]+273.15); L=C*un*w*2*np.pi*r*dx+dt*hi*B*w*2*np.pi*r*ds(1)+dt*ho*S*w*2*np.pi*r*ds(2); uu=LinearProblem(a,L,petsc_options_prefix=f'cd{n}_',petsc_options={'ksp_type':'preonly','pc_type':'lu'}).solve(); q=fem.assemble_scalar(fem.form(hi*(B-uu)*2*np.pi*r*ds(1))); bulk.append(float(B.value)-q*dt/fluidC-273.15); cc=V.tabulate_dof_coordinates(); wall.append(float((uu.x.array-273.15)[np.isclose(cc[:,1],ri)].min())); t.append((n+1)*dt/3600); un.x.array[:]=uu.x.array
    return np.array(t),np.array(bulk),np.array(wall)
tc,bc,wc=cooldown(); cross=lambda y: (tc[np.where(y<=target)[0][0]] if np.any(y<=target) else np.inf); print('FEM bulk no-touch:',cross(bc),'h; local wall no-touch:',cross(wc),'h'); plt.plot(tc,bc,label='bulk'); plt.plot(tc,wc,label='min wall'); plt.axhline(target,ls='--'); plt.legend(); plt.xlabel('h'); plt.ylabel('degC'); plt.grid(); plt.show()

## 4. Thermo-mechanical stress screen

The steady thermal field is transferred to a steel-only axisymmetric elasticity model with internal pressure and external hydrostatic pressure. This demonstrates the same handoff pattern: NeqSim sets fluid loads; FEniCSx resolves local displacement/stress.

In [ ]:
from scipy.interpolate import LinearNDInterpolator,NearestNDInterpolator
xy=Th.function_space.tabulate_dof_coordinates()[:,:2]; lin=LinearNDInterpolator(xy,Th.x.array); near=NearestNDInterpolator(xy,Th.x.array)
def ft(x):
    p=np.c_[x[0],x[1]]; y=lin(p); bad=~np.isfinite(y); y[bad]=near(p[bad]); return y
sm=mesh.create_rectangle(MPI.COMM_WORLD,np.array([[0.,ri],[localL,rs]]),[80,8],cell_type=mesh.CellType.triangle); st=tags_rect(sm,ri,rs); ds=ufl.Measure('ds',domain=sm,subdomain_data=st); dx=ufl.Measure('dx',domain=sm); VT=fem.functionspace(sm,('Lagrange',1)); TT=fem.Function(VT); TT.interpolate(ft); V=fem.functionspace(sm,('Lagrange',1,(2,))); u=ufl.TrialFunction(V); w=ufl.TestFunction(V); X=ufl.SpatialCoordinate(sm); r=X[1]; E,nu,alpha=207e9,.30,12e-6; mu=E/(2*(1+nu)); la=E*nu/((1+nu)*(1-2*nu)); I=ufl.Identity(3)
def eps(a): return ufl.as_tensor([[a[0].dx(0),.5*(a[0].dx(1)+a[1].dx(0)),0],[.5*(a[0].dx(1)+a[1].dx(0)),a[1].dx(1),0],[0,0,a[1]/r]])
def sig(a):
    ee=eps(a)-alpha*(TT-(Tb+273.15))*I; return 2*mu*ee+la*ufl.tr(ee)*I
a=ufl.inner(sig(u),eps(w))*2*np.pi*r*dx; pi=Pb*1e5; po=rhosea*g*depth; L=ufl.dot(ufl.as_vector((0.,pi)),w)*2*np.pi*r*ds(1)+ufl.dot(ufl.as_vector((0.,-po)),w)*2*np.pi*r*ds(2); fd=sm.topology.dim-1; lf=mesh.locate_entities_boundary(sm,fd,lambda x:np.isclose(x[0],0.)); dz=fem.locate_dofs_topological(V.sub(0),fd,lf); bc=fem.dirichletbc(PETSc.ScalarType(0.),dz,V.sub(0)); uh=LinearProblem(a,L,bcs=[bc],petsc_options_prefix='el_',petsc_options={'ksp_type':'preonly','pc_type':'lu'}).solve(); S=sig(uh); dev=S-ufl.tr(S)/3*I; vm=ufl.sqrt(1.5*ufl.inner(dev,dev)); V0=fem.functionspace(sm,('Discontinuous Lagrange',0)); q=ufl.TrialFunction(V0); v0=ufl.TestFunction(V0); vmh=LinearProblem(q*v0*dx,vm*v0*dx,petsc_options_prefix='vm_',petsc_options={'ksp_type':'preonly','pc_type':'lu'}).solve(); print('Maximum local von Mises:',float(vmh.x.array.max()/1e6),'MPa')

## Model hierarchy

Use NeqSim for the thermodynamic/process envelope and fast screening; use local FEM only where geometry or material gradients matter. The companion `finite_element_methods_oil_gas_neqsim.ipynb` shows the wider **Gmsh + scikit-fem + FEniCSx + PyVista** stack, including porous diffusion and wellbore/formation heat transfer.